# Basketball Team Classification — Exploration

Evaluates team classification approaches on three basketball video clips. Video clips and ground truth labels were sourced independently (sample data was not provided). A custom data pipeline was built covering YouTube clip extraction, YOLO-based player detection, and an interactive manual labeling tool in `notebooks/data_collection.ipynb`.

| Clip | Match-up | Difficulty |
|------|----------|------------|
| `clip1_easy.mp4` | Celtics vs Heat | Easy — white/green vs black/red |
| `clip2_hard.mp4` | Spurs vs Grizzlies | Hard — light blue vs dark navy |
| `clip3_edge.mp4` | Cavs vs Knicks Christmas | Edge — navy throwback vs white/orange |

**Accuracy is measured on valid single-player detections only.** Merged bounding boxes and ambiguous detections were marked `valid=False` during labeling and are excluded from all calculations.

**Before running:** upload the three `.mp4` files to `/content/`.

## Section 1 — Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless transformers accelerate torch torchvision

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ManuPrabakaran/vlm_team_classifier"
REPO_DIR = "/content/vlm_team_classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo ready at {REPO_DIR}")

In [ ]:
import cv2
import json
import numpy as np
from pathlib import Path
from ultralytics import YOLO

from src.utils import extract_frames, detect_players
from src.config import YOLO_CONFIDENCE

In [ ]:
yolo_model = YOLO("yolov8n.pt")

In [ ]:
clips = {
    "clip1_easy": {
        "video_path": "/content/clip1_easy.mp4",
        "gt_path": f"{REPO_DIR}/data/clip1_easy_ground_truth.json",
        "description": "Celtics vs Heat - contrasting jerseys (white/green vs black/red)",
        "remove_first_frame": True,
    },
    "clip2_hard": {
        "video_path": "/content/clip2_hard.mp4",
        "gt_path": f"{REPO_DIR}/data/clip2_hard_ground_truth.json",
        "description": "Spurs vs Grizzlies - similar colors (light blue vs dark navy)",
        "remove_first_frame": True,
    },
    "clip3_edge": {
        "video_path": "/content/clip3_edge.mp4",
        "gt_path": f"{REPO_DIR}/data/clip3_edge_ground_truth.json",
        "description": "Cavs vs Knicks Christmas - alternate jerseys (navy throwback vs white/orange)",
        "remove_first_frame": False,
    },
}

In [ ]:
# extract_frames returns [(frame, timestamp), ...] — strip timestamps for indexing
frames_dict = {}      # clip_name -> list of BGR numpy arrays
detections_dict = {}  # clip_name -> list of bbox lists (one per frame)
gt_dict = {}          # clip_name -> ground truth list

for clip_name, clip_info in clips.items():
    print(f"Processing {clip_name}...")

    # Extract frames at 1 FPS and strip timestamps
    frames = [f for f, _ in extract_frames(clip_info["video_path"], sample_rate=1)]

    # Run YOLO detection on each frame
    detections = [detect_players(f, yolo_model, YOLO_CONFIDENCE) for f in frames]

    # Remove the 0th extracted frame for clip1 and clip2.
    # The very first second is a pre-game shot; after removal, GT frame_idx 0
    # aligns with the next extracted frame.
    if clip_info["remove_first_frame"] and len(frames) > 1:
        frames.pop(0)
        detections.pop(0)

    frames_dict[clip_name] = frames
    detections_dict[clip_name] = detections

    with open(clip_info["gt_path"]) as f:
        gt_dict[clip_name] = json.load(f)

    print(f"  {len(frames)} video frames, {len(gt_dict[clip_name])} GT frames")

print("\nAll clips loaded.")

## Section 2 — K-Means Baseline

The baseline fits K-Means (k=2) on mean RGB jersey color from the first labeled frame, then predicts team for all subsequent detections. Label orientation is resolved by trying both 0/1 assignments and keeping the higher accuracy.

| Clip | Accuracy | Valid detections | Stability |
|------|----------|-----------------|----------|
| clip1_easy | 89.1% (82.6% in ~10% of runs) | 46 | Mostly stable — rare bad init |
| clip2_hard | 54.0% (stable) | 50 | Stable failure — colors too similar |
| clip3_edge | 90.6% or 81.2% (~50/50) | 32 | Bimodal — alternates almost every other run |

**Non-determinism**: `KMeans` has no `random_state`, producing three distinct regimes:

- **clip1** is mostly stable (89.1%) because the colors are distinct enough that initialization rarely matters — but drops to 82.6% in ~10% of runs, showing even easy cases aren't immune.
- **clip2** is stable at 54% for the opposite reason: colors are so similar that K-Means consistently fails regardless of initialization. No good local minimum exists.
- **clip3** alternates between exactly 90.6% and 81.2% on almost every other run — a bimodal distribution. K-Means has two equally-attractive local minima for this color distribution and random init decides which one it lands on.

**YOLO detection bias**: in clip2_hard, dark navy Grizzlies jerseys blend into the arena background, producing roughly a 4:1 ratio of Spurs to Grizzlies detections. This biases the K-Means centroids toward the over-represented team — detection quality and classification quality are coupled.

In [ ]:
from src.baseline import TeamClustering

In [ ]:
def evaluate_kmeans(frames, gt_data):
    """
    Fit K-Means on the first GT frame with >=2 valid non-referee player bboxes,
    then predict team for every valid non-referee detection across all GT frames.
    Tries both label orientations (0/1 swap) and returns the higher accuracy.

    Args:
        frames:  list of BGR numpy arrays indexed by GT frame_idx
        gt_data: list of dicts {frame_idx, timestamp, labels: [{bbox, team_id, valid}]}

    Returns:
        accuracy (float), n_detections (int)
    """
    # Find first frame with >=2 valid player bboxes (exclude referees from fit)
    fit_frame, fit_bboxes = None, []
    for entry in gt_data:
        player_bboxes = [
            label["bbox"]
            for label in entry["labels"]
            if label["valid"] and label["team_id"] != -1
        ]
        if len(player_bboxes) >= 2:
            fit_frame = frames[entry["frame_idx"]]
            fit_bboxes = player_bboxes
            break

    assert fit_frame is not None, (
        "No GT frame with >=2 valid player bboxes found. "
        "Check that frame indices align after first-frame removal."
    )

    tc = TeamClustering()
    tc.fit(fit_frame, fit_bboxes)

    predictions, ground_truth_labels = [], []
    for entry in gt_data:
        frame = frames[entry["frame_idx"]]
        for label in entry["labels"]:
            if not label["valid"] or label["team_id"] == -1:
                continue  # skip invalid boxes and referees
            predictions.append(int(tc.predict_team(frame, label["bbox"])))
            ground_truth_labels.append(label["team_id"])

    preds = np.array(predictions)
    gt    = np.array(ground_truth_labels)

    acc_normal  = float(np.mean(preds == gt))
    acc_flipped = float(np.mean((1 - preds) == gt))
    return max(acc_normal, acc_flipped), len(predictions)

In [ ]:
kmeans_results = {}

for clip_name, clip_info in clips.items():
    acc, n = evaluate_kmeans(frames_dict[clip_name], gt_dict[clip_name])
    kmeans_results[clip_name] = {
        "accuracy": round(acc, 3),
        "n_detections": n,
        "description": clip_info["description"],
    }
    print(f"{clip_name}: {acc:.1%}  ({n} detections)")

### Observations on K-Means Stability

Something interesting came up while running the baseline. The `TeamClustering` class initializes `KMeans` without a `random_state`, which means every run gets a different random starting point for the cluster centroids. In practice this turns out to matter a lot more than expected.

For **clip3_edge**, the accuracy flips between 90.6% and 81.2% on almost every other run. That is not just noise, it is the model landing on two completely different cluster solutions depending on where initialization starts. For **clip1_easy**, the result is usually 89.1% but falls to 82.6% about 10% of the time. Even with visually distinct jerseys, there is still a bad local minimum that the model occasionally gets stuck in.

The most paradoxical result is **clip2_hard**: it is actually the *most* stable across runs, locking in at 54.0% every time. That stability is not a good thing though. It just means the colors are so similar that K-Means finds the same wrong answer no matter where it starts.

This connects to color cluster separation. When colors are very distinct (clip1), initialization rarely changes the outcome. When colors are completely ambiguous (clip2), initialization also does not matter because there is no correct answer to find. clip3 sits right in the middle zone where starting centroids actually determine which of two valid-looking solutions the model converges to.

To get a more reliable picture, the cell below runs K-Means 10 times per clip and reports mean, std, min, and max accuracy.

In [ ]:
n_runs = 10
stability_results = {}

for clip_name in clips.keys():
    accs = []
    frames = frames_dict[clip_name]
    gt_data = gt_dict[clip_name]
    for _ in range(n_runs):
        acc, n = evaluate_kmeans(frames, gt_data)
        accs.append(acc)
    stability_results[clip_name] = {
        "mean": float(np.mean(accs)),
        "std": float(np.std(accs)),
        "min": float(np.min(accs)),
        "max": float(np.max(accs)),
        "n_detections": n,
    }
    print(
        f"{clip_name}: mean={np.mean(accs):.1%} +/- {np.std(accs):.1%} "
        f"(min={np.min(accs):.1%}, max={np.max(accs):.1%})"
    )

In [ ]:
metrics_path = Path(REPO_DIR) / "results" / "metrics.json"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
else:
    metrics = {}

metrics["baseline_kmeans"] = kmeans_results

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved to {metrics_path}")
print(json.dumps(metrics["baseline_kmeans"], indent=2))

## Section 3 — Model Comparison

The K-Means baseline fails in two distinct ways: (1) it relies on mean RGB color, which collapses when jerseys are similar (clip2_hard, 54%) or lighting varies, and (2) it is sensitive to random initialization on edge cases (clip3_edge). The goal here is to test whether richer visual representations fix these failure modes.

| Model | ID | Approach | Why evaluate |
|-------|----|----------|--------------|
| **SigLIP** | `google/siglip-base-patch16-224` | Embed crops → cluster | Fast, strong image-text alignment, ~350 MB |
| **CLIP** | `openai/clip-vit-base-patch32` | Embed crops → cluster | Widely benchmarked zero-shot baseline |
| **Florence-2** | `microsoft/Florence-2-base` | Prompted classification | Rich visual grounding, region-level reasoning |
| **Qwen2-VL** | `Qwen/Qwen2-VL-2B-Instruct` | Prompted classification | Production stack model; strongest reasoning |

**Embedding strategy (SigLIP, CLIP):** extract per-player crops → get image embeddings → build team prototype centroids from the first GT frame → classify by cosine distance. This replaces mean RGB with a richer feature vector that encodes texture, logo shape, and number patterns — not just color — making it more robust to the similar-color failure mode.

**Prompting strategy (Florence-2, Qwen2-VL):** send each player crop with a structured prompt describing both teams' jersey colors established from the tipoff frame. These models can reason about visual context beyond raw pixel statistics, but have higher latency and memory cost.

### 3.1 SigLIP — Embedding + Cosine Clustering

In [ ]:
# TODO: SigLIP implementation
#
# from transformers import AutoProcessor, AutoModel
# from src.config import MODEL_SIGLIP
# from src.utils import extract_siglip_embedding, load_siglip_model
# import torch
#
# device = "cuda" if torch.cuda.is_available() else "cpu"
# siglip_model, siglip_processor = load_siglip_model(device)
#
# def get_siglip_embedding(crop_bgr):
#     return extract_siglip_embedding(crop_bgr, siglip_model, siglip_processor, device)
#
# def evaluate_embedding_model(embed_fn, frames_dict, gt_dict):
#     """Build team prototypes from first GT frame, classify remaining frames."""
#     ...
#
# siglip_results = evaluate_embedding_model(get_siglip_embedding, frames_dict, gt_dict)
print("SigLIP — placeholder, implementation pending.")

### 3.2 CLIP — Embedding + Cosine Clustering

In [ ]:
# TODO: CLIP implementation
#
# from transformers import CLIPProcessor, CLIPModel
# import torch
# from PIL import Image
#
# CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
# clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()
#
# def get_clip_embedding(crop_bgr):
#     image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
#     inputs = clip_processor(images=image, return_tensors="pt").to(device)
#     with torch.no_grad():
#         feats = clip_model.get_image_features(**inputs)[0].cpu().numpy()
#     return feats / np.linalg.norm(feats)
#
# clip_results = evaluate_embedding_model(get_clip_embedding, frames_dict, gt_dict)
print("CLIP — placeholder, implementation pending.")

### 3.3 Florence-2 — Direct Prompted Classification

In [ ]:
# TODO: Florence-2 implementation
#
# from transformers import AutoProcessor, AutoModelForCausalLM
# import torch
# from PIL import Image
#
# FLORENCE_MODEL_ID = "microsoft/Florence-2-base"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# florence_processor = AutoProcessor.from_pretrained(FLORENCE_MODEL_ID, trust_remote_code=True)
# florence_model = AutoModelForCausalLM.from_pretrained(
#     FLORENCE_MODEL_ID, trust_remote_code=True
# ).to(device).eval()
#
# def classify_with_florence(crop_bgr, team0_desc, team1_desc):
#     image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
#     prompt = (
#         f"<CLASSIFICATION> Team 0 wears {team0_desc} jerseys. "
#         f"Team 1 wears {team1_desc} jerseys. "
#         "Which team does this basketball player belong to?"
#     )
#     inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(device)
#     with torch.no_grad():
#         output_ids = florence_model.generate(**inputs, max_new_tokens=10)
#     response = florence_processor.decode(output_ids[0], skip_special_tokens=True)
#     return 0 if "0" in response else 1
#
# florence_results = evaluate_prompted_model(classify_with_florence, frames_dict, gt_dict)
print("Florence-2 — placeholder, implementation pending.")

### 3.4 Qwen2-VL — Direct Prompted Classification

In [ ]:
# TODO: Qwen2-VL implementation
#
# from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
# from src.config import MODEL_QWEN
# from src.utils import load_qwen_model
# import torch
# from PIL import Image
#
# device = "cuda" if torch.cuda.is_available() else "cpu"
# qwen_model, qwen_processor = load_qwen_model(device)
#
# def classify_with_qwen(crop_bgr, team0_desc, team1_desc):
#     image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
#     prompt = (
#         f"Team 0 wears {team0_desc} jerseys. "
#         f"Team 1 wears {team1_desc} jerseys. "
#         "Look at this basketball player crop. "
#         "Which team does this player belong to? Reply with only '0' or '1'."
#     )
#     messages = [{
#         "role": "user",
#         "content": [{"type": "image"}, {"type": "text", "text": prompt}]
#     }]
#     text = qwen_processor.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True
#     )
#     inputs = qwen_processor(
#         text=[text], images=[image], return_tensors="pt"
#     ).to(device)
#     with torch.no_grad():
#         output_ids = qwen_model.generate(**inputs, max_new_tokens=5)
#     response = qwen_processor.decode(output_ids[0], skip_special_tokens=True).strip()
#     return 0 if "0" in response else 1
#
# qwen_results = evaluate_prompted_model(classify_with_qwen, frames_dict, gt_dict)
print("Qwen2-VL — placeholder, implementation pending.")